# Day 8-9：高级 RAG 架构

RAG（Retrieval-Augmented Generation）是 LLM 应用最核心的技术之一。
基础 RAG 的流程很简单：检索文档 → 拼到 Prompt → 让 LLM 生成答案。

但实际工程中，基础 RAG 的效果往往很差：
- 分块太大 → 语义模糊，检索不准
- 分块太小 → 上下文不足，答案不完整
- 只用向量检索 → 漏掉关键词完全匹配的文档

这两天我们深入 RAG 的每个环节，掌握**分块策略**和**检索优化**。

## 一、Chunking 策略详解

分块是 RAG 的第一步，也是最容易被忽视的一步。分块质量直接决定检索质量。

### 1.1 固定分块（Fixed-size Chunking）

最简单的策略：按固定 Token 数切分，比如每 512 Token 一块。

```
原始文档：[Token1, Token2, ..., Token2000]
切分结果：
  Chunk 1: [Token1 ... Token512]
  Chunk 2: [Token513 ... Token1024]
  Chunk 3: [Token1025 ... Token1536]
  Chunk 4: [Token1537 ... Token2000]
```

**问题**：会切断句子、段落，甚至把一个完整的概念劈成两半。

### 1.2 递归分块（Recursive Chunking）

LangChain 默认策略。先按大分隔符（如双换行）切，切完还太大就按小分隔符（单换行、句号）再切。

```
先按 \n\n 切 → 按 \n 切 → 按 。 切 → 按字符切
```

In [ ]:
# ===== 演示：三种分块策略的对比 =====
# 核心理解：不同策略切出来的块，语义完整性差异巨大

sample_text = """LangGraph 是 LangChain 团队开发的状态图框架。

它用有向图来管理 Agent 的工作流。节点是处理函数，边是流转规则。
状态（State）是所有节点共享的数据容器，用 TypedDict 定义。

条件边是 LangGraph 最强大的特性。通过路由函数，图可以根据运行时状态
动态选择下一个节点。这比 Chain 的线性流程灵活得多。

Human-in-the-loop 是另一个关键特性。通过 interrupt() 函数，
Agent 可以在执行危险操作前暂停，等待人工确认后再继续。

Checkpointer 负责持久化状态。每次执行一步，系统自动保存快照。
支持 Time Travel：回退到任意历史状态重新执行。"""

# ===== 策略1：固定分块 =====
def fixed_chunking(text, chunk_size=100):
    """按固定字符数切分，不考虑语义"""
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])
    return chunks

# ===== 策略2：按段落分块 =====
def paragraph_chunking(text):
    """按双换行切分，保留段落完整性"""
    chunks = text.split("\n\n")
    return [c.strip() for c in chunks if c.strip()]

# ===== 策略3：递归分块（LangChain 风格）=====
def recursive_chunking(text, max_size=150, separators=None):
    """递归切分：先大分隔符，再小分隔符"""
    if separators is None:
        separators = ["\n\n", "\n", "。", "，", " "]
    
    if len(text) <= max_size:
        return [text]
    
    for sep in separators:
        if sep in text:
            parts = text.split(sep)
            chunks = []
            current = ""
            for part in parts:
                if len(current) + len(part) <= max_size:
                    current += part + sep
                else:
                    if current:
                        chunks.append(current.strip())
                    current = part + sep
            if current:
                chunks.append(current.strip())
            return chunks
    
    return [text]

# 对比三种策略
print("=" * 60)
print("策略1: 固定分块（每100字符）")
print("=" * 60)
for i, chunk in enumerate(fixed_chunking(sample_text, 100)):
    print(f"\nChunk {i+1} ({len(chunk)}字): {chunk[:80]}...")

print("\n" + "=" * 60)
print("策略2: 按段落分块")
print("=" * 60)
for i, chunk in enumerate(paragraph_chunking(sample_text)):
    print(f"\nChunk {i+1} ({len(chunk)}字): {chunk[:80]}...")

print("\n" + "=" * 60)
print("策略3: 递归分块（最大150字符）")
print("=" * 60)
for i, chunk in enumerate(recursive_chunking(sample_text, 150)):
    print(f"\nChunk {i+1} ({len(chunk)}字): {chunk[:80]}...")

### 1.3 语义分块（Semantic Chunking）

最高级的分块策略：用 Embedding 相似度判断哪里该切。

核心思想：相邻句子的语义通常相近。如果两个句子的 Embedding 相似度骤降，
说明话题发生了转换，这里应该切开。

```
句子1: "LangGraph 用有向图管理工作流" → Embedding
句子2: "节点是处理函数" → Embedding → 相似度 0.85 → 不切
句子3: "条件边是最强大的特性" → Embedding → 相似度 0.78 → 不切
句子4: "Human-in-the-loop 是另一个特性" → Embedding → 相似度 0.45 → 切！
```

实现方式：计算相邻句子的余弦相似度，低于阈值时切开。

In [ ]:
# ===== 语义分块的实现原理 =====
# 注意：实际使用需要安装 sentence-transformers
# pip install sentence-transformers

import numpy as np

def cosine_similarity(a, b):
    """计算两个向量的余弦相似度"""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# 模拟：用随机向量代替真实 Embedding，演示分块逻辑
# 实际项目中，这里用的是 sentence-transformers 的 encode()

sentences = [
    "LangGraph 用有向图管理工作流",
    "节点是处理函数，边是流转规则",
    "状态是所有节点共享的数据容器",
    "条件边是最强大的特性",       # 话题：图的特性
    "路由函数根据状态选择节点",     # 话题：图的特性
    "Human-in-the-loop 是另一个特性",  # 话题转换！
    "interrupt 函数可以暂停执行",
    "Checkpointer 负责持久化状态",   # 话题转换！
    "支持 Time Travel 回退历史状态",
]

# 模拟 Embedding 相似度（实际中用模型计算）
# 同话题的句子相似度高，不同话题相似度低
similarities = [0.0, 0.85, 0.82, 0.78, 0.80, 0.42, 0.75, 0.38, 0.82]

def semantic_chunking(sentences, similarities, threshold=0.5):
    """
    语义分块：相似度低于阈值时切开
    """
    chunks = []
    current_chunk = [sentences[0]]
    
    for i in range(1, len(sentences)):
        sim = similarities[i]
        if sim < threshold:
            # 话题转换，切开
            chunks.append(current_chunk)
            current_chunk = [sentences[i]]
            print(f"  切分点: [{i}] 相似度={sim:.2f} < {threshold}")
        else:
            current_chunk.append(sentences[i])
    
    if current_chunk:
        chunks.append(current_chunk)
    
    return chunks

print("语义分块结果（阈值=0.5）：")
print("=" * 50)
chunks = semantic_chunking(sentences, similarities, threshold=0.5)
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}:")
    for s in chunk:
        print(f"  - {s}")

print("\n" + "=" * 50)
print("关键洞察：")
print("语义分块自动识别了话题转换点，切出来的块语义更完整")
print("缺点：需要调用 Embedding 模型，分块阶段就有计算成本")

## 二、Parent-Child Chunking

这是工程中最常用的分块策略之一。核心思想：

**检索时用小块（Child），返回时用大块（Parent）。**

```
Parent Chunk（500-1000 Token）：完整的上下文
┌──────────────────────────────────────┐
│ LangGraph 是 LangChain 团队开发的    │
│ 状态图框架。它用有向图来管理 Agent    │
│ 的工作流。节点是处理函数，边是流转    │
│ 规则。状态是所有节点共享的数据容器。  │
└──────────────────────────────────────┘
         ↓ 切成小块
Child Chunk 1（50-100 Token）：精确匹配
┌─────────────────┐
│ LangGraph 是     │
│ 状态图框架       │
└─────────────────┘
Child Chunk 2（50-100 Token）：精确匹配
┌─────────────────┐
│ 节点是处理函数   │
│ 边是流转规则     │
└─────────────────┘
```

**为什么这样设计？**
- Child 小 → Embedding 语义精准，检索召回率高
- Parent 大 → 包含完整上下文，LLM 生成答案时信息充足
- 检索命中 Child → 返回对应的 Parent 给 LLM

In [ ]:
# ===== Parent-Child Chunking 实现 =====

class ParentChildChunker:
    """
    Parent-Child 分块器
    
    核心逻辑：
    1. 先按 Parent 大小切分（保留完整上下文）
    2. 每个 Parent 再切成 Child（用于精确检索）
    3. 建立 Parent ↔ Child 的映射关系
    """
    
    def __init__(self, parent_size=500, child_size=100):
        self.parent_size = parent_size
        self.child_size = child_size
    
    def chunk(self, text):
        """返回 (parents, children, child_to_parent) 映射"""
        # Step 1: 切 Parent
        parents = []
        for i in range(0, len(text), self.parent_size):
            parent = text[i:i+self.parent_size]
            if parent.strip():
                parents.append(parent)
        
        # Step 2: 每个 Parent 切成 Child
        children = []
        child_to_parent = {}  # child_index → parent_index
        
        for p_idx, parent in enumerate(parents):
            for j in range(0, len(parent), self.child_size):
                child = parent[j:j+self.child_size]
                if child.strip():
                    c_idx = len(children)
                    children.append(child)
                    child_to_parent[c_idx] = p_idx
        
        return parents, children, child_to_parent

# 演示
text = """LangGraph 是 LangChain 团队开发的状态图框架。它用有向图来管理 Agent 的工作流。
节点是处理函数，边是流转规则，状态是所有节点共享的数据容器。
条件边是 LangGraph 最强大的特性。通过路由函数，图可以根据运行时状态动态选择下一个节点。
Human-in-the-loop 是另一个关键特性。通过 interrupt() 函数，Agent 可以在执行危险操作前暂停。
Checkpointer 负责持久化状态。每次执行一步，系统自动保存快照。支持 Time Travel 回退历史。
在实际 Agent 开发中，这些特性组合使用，构建出可靠的、可中断的、可回溯的工作流。"""

chunker = ParentChildChunker(parent_size=200, child_size=60)
parents, children, mapping = chunker.chunk(text)

print("Parent-Child 分块结果：")
print("=" * 60)
print(f"Parent 数量: {len(parents)}")
print(f"Child 数量: {len(children)}")
print(f"\n映射关系（Child → Parent）：")
for c_idx, p_idx in mapping.items():
    print(f"  Child {c_idx} → Parent {p_idx}")

print("\n" + "=" * 60)
print("Parent 块（大，给 LLM 看的上下文）：")
for i, p in enumerate(parents):
    print(f"\nParent {i} ({len(p)}字): {p[:60]}...")

print("\n" + "=" * 60)
print("Child 块（小，用于检索匹配）：")
for i, c in enumerate(children):
    print(f"  Child {i} → Parent {mapping[i]}: {c[:40]}...")

print("\n" + "=" * 60)
print("\n核心流程：")
print("1. 用户 Query → Embedding")
print("2. 和所有 Child 的 Embedding 计算相似度")
print("3. 命中 Child 2 → 查映射 → 返回 Parent 1")
print("4. LLM 拿到 Parent 1 的完整上下文生成答案")

## 三、Sentence Window Retrieval

比 Parent-Child 更细粒度的方案：**用单句检索，返回窗口上下文**。

```
原始文档：
  句1: "LangGraph 是状态图框架"
  句2: "它用有向图管理工作流"       ← 检索命中这句
  句3: "节点是处理函数"
  句4: "边是流转规则"
  句5: "状态是共享数据容器"

窗口大小 = 2，返回：
  [句1, 句2, 句3, 句4]   ← 命中句 + 前后各2句
```

**窗口大小的取舍：**
- 太大（如10句）→ 噪声多，稀释关键信息
- 太小（如1句）→ 上下文不足
- 经验值：3-5句，大多数场景够用

In [ ]:
# ===== Sentence Window Retrieval 实现 =====

class SentenceWindowRetriever:
    """
    句子窗口检索器
    
    核心逻辑：
    1. 把文档切成句子，每个句子单独 Embedding
    2. 检索时用单句 Embedding 匹配
    3. 返回命中句子 + 前后 N 句的窗口
    """
    
    def __init__(self, window_size=2):
        self.window_size = window_size
        self.sentences = []
    
    def index(self, text):
        """把文档切成句子并建立索引"""
        # 简单按句号切分（实际项目中用更精确的分句器）
        self.sentences = [s.strip() for s in text.split("。") if s.strip()]
        return self.sentences
    
    def retrieve(self, matched_idx):
        """根据命中句子的索引，返回窗口内的所有句子"""
        start = max(0, matched_idx - self.window_size)
        end = min(len(self.sentences), matched_idx + self.window_size + 1)
        
        window = self.sentences[start:end]
        context = {
            "matched_sentence": self.sentences[matched_idx],
            "window_sentences": window,
            "window_range": (start, end),
            "full_context": "。".join(window) + "。"
        }
        return context

# 演示
doc = """LangGraph 是状态图框架。它用有向图管理工作流。节点是处理函数。
边是流转规则。状态是共享数据容器。条件边是最强大的特性。
路由函数根据状态选择节点。Human-in-the-loop 是另一个特性。
interrupt 函数可以暂停执行。Checkpointer 负责持久化状态。
支持 Time Travel 回退历史。在实际 Agent 开发中这些特性组合使用。"""

retriever = SentenceWindowRetriever(window_size=2)
sentences = retriever.index(doc)

print(f"文档共切分为 {len(sentences)} 个句子：")
for i, s in enumerate(sentences):
    print(f"  [{i}] {s}")

# 模拟：用户查询 "什么是条件边"，命中句子5
print("\n" + "=" * 60)
print("查询: '什么是条件边' → 命中句子 [5]")
print("=" * 60)
result = retriever.retrieve(5)
print(f"\n命中句: {result['matched_sentence']}")
print(f"窗口范围: [{result['window_range'][0]}, {result['window_range'][1]})")
print(f"返回上下文: {result['full_context']}")

print("\n" + "=" * 60)
print("对比：不同窗口大小的效果")
print("=" * 60)
for ws in [0, 1, 2, 4]:
    r = SentenceWindowRetriever(window_size=ws)
    r.index(doc)
    ctx = r.retrieve(5)
    print(f"\n窗口={ws}: {ctx['full_context'][:80]}...")

## 四、元数据过滤

给每个 Chunk 附加元数据（Metadata），检索时可以先过滤再搜索。

```
Chunk: "LangGraph 是状态图框架"
Metadata:
  - source: "langgraph_docs.md"
  - page: 1
  - section: "概述"
  - date: "2024-01"
  - category: "技术文档"
```

**元数据过滤的典型场景：**
- 用户问"2024年的政策" → 先过滤 `date >= 2024`
- 用户问"技术文档" → 先过滤 `category == '技术文档'`
- 多知识库场景 → 先过滤 `source == '产品手册'`

这比纯向量检索精准得多——先缩小范围，再做语义匹配。

In [ ]:
# ===== 元数据过滤的实现思路 =====

from dataclasses import dataclass, field
from typing import Optional

@dataclass
class Chunk:
    """带元数据的文档块"""
    content: str
    source: str = ""
    page: int = 0
    section: str = ""
    date: str = ""
    category: str = ""
    embedding: list = field(default_factory=list)

# 模拟：构建带元数据的 Chunk 索引
chunks = [
    Chunk(content="LangGraph 是状态图框架", source="docs.md", section="概述", date="2024-01", category="技术"),
    Chunk(content="节点是处理函数", source="docs.md", section="核心概念", date="2024-01", category="技术"),
    Chunk(content="条件边实现路由", source="docs.md", section="核心概念", date="2024-01", category="技术"),
    Chunk(content="产品定价方案A", source="pricing.md", section="方案A", date="2024-06", category="商务"),
    Chunk(content="产品定价方案B", source="pricing.md", section="方案B", date="2025-01", category="商务"),
    Chunk(content="interrupt 实现 HITL", source="tutorial.md", section="高级特性", date="2024-06", category="教程"),
]

def metadata_filter(chunks, **filters):
    """先按元数据过滤，再返回结果"""
    result = []
    for chunk in chunks:
        match = True
        for key, value in filters.items():
            if getattr(chunk, key, None) != value:
                match = False
                break
        if match:
            result.append(chunk)
    return result

# 演示：不同过滤条件
print("所有 Chunk：")
for c in chunks:
    print(f"  [{c.source}] {c.content} (category={c.category}, date={c.date})")

print("\n" + "=" * 50)
print("过滤: category='技术'")
for c in metadata_filter(chunks, category="技术"):
    print(f"  → {c.content}")

print("\n过滤: date='2024-06'")
for c in metadata_filter(chunks, date="2024-06"):
    print(f"  → {c.content}")

print("\n过滤: source='pricing.md'")
for c in metadata_filter(chunks, source="pricing.md"):
    print(f"  → {c.content}")

print("\n" + "=" * 50)
print("实际 RAG 流程：")
print("1. 用户 Query → 先判断是否需要元数据过滤")
print("2. 如果需要 → LLM 提取过滤条件（如 category='技术'）")
print("3. 先过滤 → 再对过滤后的 Chunk 做向量检索")
print("4. 返回 Top-K 结果")

## 五、向量检索基础

分块完成后，需要把每个 Chunk 转成 Embedding 向量，存入向量数据库。

### Embedding 模型选择

| 模型 | 维度 | 特点 |
|------|------|------|
| text-embedding-3-small | 1536 | OpenAI，便宜但效果一般 |
| BGE-M3 | 1024 | BAAI，中文效果最好，开源 |
| GTE-Qwen2 | 1536 | 阿里，中文好，开源 |
| jina-embeddings-v3 | 1024 | 多语言，开源 |

中文项目首选 **BGE-M3** 或 **GTE-Qwen2**。

In [ ]:
# ===== 向量检索的基本原理 =====
# 用 numpy 模拟 Embedding 检索过程

import numpy as np

def cosine_similarity(a, b):
    """余弦相似度"""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# 模拟：3 个 Chunk 的 Embedding（实际中维度是 1024 或 1536）
np.random.seed(42)
dim = 16  # 简化为16维

chunk_embeddings = {
    "LangGraph 是状态图框架": np.random.randn(dim),
    "条件边实现动态路由": np.random.randn(dim),
    "产品定价方案A是99元": np.random.randn(dim),
}

# 模拟 Query 的 Embedding
query_embedding = np.random.randn(dim)

# 计算 Query 和每个 Chunk 的相似度
print("向量检索过程：")
print("=" * 50)
print(f"Query Embedding (dim={dim}): {query_embedding[:4]}...")
print()

scores = []
for text, emb in chunk_embeddings.items():
    sim = cosine_similarity(query_embedding, emb)
    scores.append((text, sim))
    print(f"  '{text}' → 相似度: {sim:.4f}")

# 按相似度排序
scores.sort(key=lambda x: x[1], reverse=True)
print(f"\nTop-1 结果: '{scores[0][0]}' (score={scores[0][1]:.4f})")

print("\n" + "=" * 50)
print("核心要点：")
print("1. Embedding 把文本变成向量（高维空间中的点）")
print("2. 语义相近的文本，向量距离更近")
print("3. 检索 = 找到和 Query 向量最相似的 Chunk 向量")
print("4. 实际中用 FAISS / Milvus / ChromaDB 做高效的向量检索")

## 六、完整 RAG Pipeline 代码框架

把上面学到的所有策略组合起来，构建一个完整的 RAG 系统框架。

这个框架包含：
1. 文档加载 → 分块（Parent-Child）→ Embedding → 存储
2. 用户查询 → 向量检索 → 返回 Parent 上下文 → LLM 生成答案

In [ ]:
# ===== 完整 RAG Pipeline 框架 =====
# 注意：这是一个教学框架，展示完整流程
# 实际项目中，向量存储用 FAISS/ChromaDB，Embedding 用 BGE-M3

import numpy as np
from typing import Optional


class SimpleRAG:
    """
    简化版 RAG 系统
    
    完整流程：
    文档 → Parent-Child 分块 → Embedding → 向量存储
    查询 → Embedding → 向量检索 → 返回 Parent → LLM 生成
    """
    
    def __init__(self, parent_size=300, child_size=80):
        self.parent_size = parent_size
        self.child_size = child_size
        self.parents = []        # Parent 块列表
        self.children = []       # Child 块列表
        self.child_to_parent = {} # Child → Parent 映射
        self.child_embeddings = [] # Child 的 Embedding
        self.dim = 16            # Embedding 维度（演示用）
    
    def _mock_embed(self, text):
        """模拟 Embedding（实际中用 BGE-M3 等模型）"""
        np.random.seed(hash(text) % 2**31)
        return np.random.randn(self.dim)
    
    def index(self, documents: list[str]):
        """索引文档：分块 + Embedding"""
        for doc in documents:
            # 1. 切 Parent
            for i in range(0, len(doc), self.parent_size):
                parent = doc[i:i+self.parent_size]
                if not parent.strip():
                    continue
                p_idx = len(self.parents)
                self.parents.append(parent)
                
                # 2. 切 Child
                for j in range(0, len(parent), self.child_size):
                    child = parent[j:j+self.child_size]
                    if not child.strip():
                        continue
                    c_idx = len(self.children)
                    self.children.append(child)
                    self.child_to_parent[c_idx] = p_idx
                    # 3. 计算 Embedding
                    self.child_embeddings.append(self._mock_embed(child))
        
        self.child_embeddings = np.array(self.child_embeddings)
        print(f"索引完成: {len(self.parents)} 个 Parent, {len(self.children)} 个 Child")
    
    def retrieve(self, query: str, top_k: int = 3) -> list[str]:
        """检索：Query → 向量相似度 → 返回 Parent 上下文"""
        query_emb = self._mock_embed(query)
        
        # 计算所有 Child 的相似度
        scores = []
        for i, child_emb in enumerate(self.child_embeddings):
            sim = np.dot(query_emb, child_emb) / (
                np.linalg.norm(query_emb) * np.linalg.norm(child_emb)
            )
            scores.append((i, sim))
        
        # 取 Top-K 个 Child，返回对应的 Parent（去重）
        scores.sort(key=lambda x: x[1], reverse=True)
        seen_parents = set()
        results = []
        
        for c_idx, score in scores[:top_k]:
            p_idx = self.child_to_parent[c_idx]
            if p_idx not in seen_parents:
                seen_parents.add(p_idx)
                results.append(self.parents[p_idx])
        
        return results

# 演示
documents = [
    "LangGraph 是 LangChain 团队开发的状态图框架。它用有向图来管理 Agent 的工作流。节点是处理函数，边是流转规则，状态是所有节点共享的数据容器。",
    "条件边是 LangGraph 最强大的特性。通过路由函数，图可以根据运行时状态动态选择下一个节点。Human-in-the-loop 通过 interrupt() 函数实现。",
    "RAG 系统的核心流程：文档分块、Embedding 向量化、向量检索、上下文注入、LLM 生成答案。分块策略决定检索质量。",
]

rag = SimpleRAG(parent_size=150, child_size=50)
rag.index(documents)

# 检索测试
print("\n" + "=" * 50)
print("检索测试")
print("=" * 50)

query = "什么是条件边"
print(f"\nQuery: {query}")
results = rag.retrieve(query, top_k=2)
for i, r in enumerate(results):
    print(f"\nResult {i+1}: {r[:80]}...")

print("\n" + "=" * 50)
print("\n注意：这里用的是模拟 Embedding，结果是随机的")
print("实际项目中用 BGE-M3 等真实 Embedding 模型，语义匹配会非常精准")

## 今日总结

今天深入了 RAG 系统最核心的两个环节：分块和检索。

| 策略 | 适用场景 | 优点 | 缺点 |
|------|---------|------|------|
| 固定分块 | 快速原型 | 实现简单 | 切断语义 |
| 递归分块 | 通用场景 | 平衡语义和大小 | 不够精确 |
| 语义分块 | 高质量需求 | 语义最完整 | 计算成本高 |
| Parent-Child | 生产环境首选 | 检索精准+上下文充足 | 索引较大 |
| Sentence Window | 长文档精确定位 | 细粒度检索 | 窗口大小难调 |

**明天学什么：** Rerank（精排）和混合检索——当向量检索不够准的时候，怎么提升检索质量。

**写在简历上的要点：**
"熟悉 RAG 系统的分块策略（Parent-Child、Sentence Window、语义分块），能根据业务场景选择合适的分块方案，优化检索质量。